# Try the personal-card-agent agent

Two ways to try this repo's agent from a notebook:

- **Option A - quick local invoke.** Builds the agent in-process with the
  OSS `deepagents` library, reusing this repo's `instructions.md` and
  `middleware/` (customer-context seeding + tool-call logging). No `mda`
  CLI or server needed - fastest way to iterate.
- **Option B - talk to the real managed agent.** Connects to the agent
  exactly as `agent.py` defines it, served locally by `mda dev .`.
- **Option C - test the data-visualization chart skill.** Builds the agent
  exactly as `server.py` does (same `backend`/`skills` wiring), needs a
  fuller set of env vars than A/B (see that section).
- **Option D - test the transfer-form skill.** Same wiring as Option C, a
  different skill/`ui["type"]` (`transfer_form` -- proposing a paydown for
  the customer's own credit card debt).
- **Option E - test the spending-jar skill.** Same wiring again, a third
  skill/`ui["type"]` pair (`create_jar`/`edit_jar` -- proposing a new or
  changed spending jar).
- **Option J - test `POST /jar-distribute` and `POST /jar-rebalance`.** The two endpoints for screens that need
  no chat (share unallocated money into jars / move balance from one jar to others). Called
  in-process with `TestClient(server.app)`, no conversation involved.
- **Option K - test the jar flows in chat.** The same jar contract through
  `POST /chat`: moving a category between two jars, putting unallocated money into a jar
  (`distribute_amount`), and a jar that does not exist yet (no `ui`).
- **Option L - test the `clarify-options` skill.** Forces a genuinely ambiguous
  situation (two similarly-named jars) so the agent asks with `clarify_options`
  buttons instead of an open question, then simulates the customer tapping one.
- **Option M - test putting an amount into a jar.** "Chuyển 300 nghìn vào hũ Tiết kiệm" straight through the agent
  (`distribute_amount`, skill `distribute-amount`, the money comes from the unallocated pool) -- no endpoint, no `/chat` route.
- **Option N - test moving money from one jar to others.** "Chuyển 500 nghìn từ hũ A sang hũ B" straight through the agent
  (`rebalance_jars` with `from_jar_id` + `allocations`, skill `rebalance-jars`) -- no endpoint, no `/chat` route.


In [1]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Notebook runs from notebooks/, but the agent's code (middleware/, data/,
# instructions.md, .env) lives one level up at the repo root.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / ".env")

import os

assert os.environ.get("AI_PLATFORM_API_KEY"), (
    f"Set AI_PLATFORM_API_KEY in {REPO_ROOT / '.env'} first"
)

from langchain_openai import ChatOpenAI

from text_utils import extract_final_text

# Same GreenNode VNG Cloud MaaS endpoint/model server.py's _default_model()
# uses -- kept in sync manually since this notebook predates that switch and
# Option A/checklist below build their own agent without importing server.py
# (importing it would also require SERVER_API_KEY/MEMORY_ID/
# USER_PREFERENCE_STRATEGY_ID just for a quick local invoke -- see Option C).
LLM_BASE_URL = "https://maas-llm-aiplatform-hcm.api.vngcloud.vn/v1"
LLM_MODEL = os.environ.get("LLM_MODEL") or "z-ai/glm-5.2-hackathon"


def default_model() -> ChatOpenAI:
    return ChatOpenAI(base_url=LLM_BASE_URL, api_key=os.environ["AI_PLATFORM_API_KEY"], model=LLM_MODEL)


In [4]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

## Option A - quick local invoke (no `mda dev` needed)

In [ ]:
from deepagents import create_deep_agent

from middleware.audit import log_tool_calls
from middleware.seed_customer_context import SeedCustomerContext

instructions = (REPO_ROOT / "instructions.md").read_text()

local_agent = create_deep_agent(
    model=default_model(),
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
)

result = local_agent.invoke(
    {
        "messages": [
            {"role": "user", "content": "Thống kê số tiền tôi đã chi tiêu vào ăn uống"}
        ]
    }
)
print(extract_final_text(result["messages"][-1].content))

## Option B - talk to the managed agent via `mda dev`

In a separate terminal, from the repo root, run:

```bash
mda dev .
```

It prints the local Agent Server URL (default `http://127.0.0.1:2024`) and
the graph/assistant name to use below - it's usually the agent `name` from
`agent.py` (`"personal-card-agent"`), but check the printed Studio URL /
server logs and adjust `ASSISTANT_ID` if it differs.

In [2]:
from langgraph_sdk import get_sync_client

SERVER_URL = "http://127.0.0.1:2024"
ASSISTANT_ID = "personal-card-agent"

client = get_sync_client(url=SERVER_URL)

thread = client.threads.create()
for chunk in client.runs.stream(
    thread["thread_id"],
    ASSISTANT_ID,
    input={
        "messages": [
            {"role": "user", "content": "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?"}
        ]
    },
    stream_mode="values",
):
    if chunk.event == "values":
        print(extract_final_text(chunk.data["messages"][-1]["content"]))

KeyboardInterrupt: 

## Option C - test the data-visualization chart skill

Uses `server.build_agent()` directly (not a fresh `create_deep_agent(...)`
call like Option A) so the run actually exercises the same `backend=
CompositeBackend(...)`/`skills=[...]` wiring `server.py` uses in production --
a plain Option-A-style agent has no `skills=[...]`, so the skill would never
be loaded.

Importing `server` pulls in its full module-level env var checks (not just
`AI_PLATFORM_API_KEY`) -- make sure these are also set in `.env`:
`SERVER_API_KEY`, `MEMORY_ID`, `USER_PREFERENCE_STRATEGY_ID` (the last one
via `tools/memory_tools.py`, imported by `server.py`).

The question below explicitly asks for a `biểu đồ` (chart), echoing the
`data-visualization` skill's own description word-for-word -- the closest
thing to a guaranteed trigger this design allows (the model still decides,
there's no enforced schema). A missing/malformed `ui-json` block just
yields `ui is None`, it never breaks the call.

**Note (2026-09-03):** the response field was renamed from `chart` to `ui`
(a shared envelope for every widget type, discriminated by `ui["type"]`)
-- see `docs/api.md`'s breaking-change note and Option D below for the
second type, `transfer_form`.

In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver, not the real GreenNode Memory checkpointer -- otherwise
# each re-run of this cell keeps appending to the SAME real, persistent
# thread ("test-chart") instead of starting a clean conversation, which
# silently changes the model's answer on later runs (it starts "remembering"
# earlier runs). Re-running this cell always starts a fresh, isolated agent.
chart_agent = server.build_agent(checkpointer=InMemorySaver())

result = chart_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Phân tích và vẽ biểu đồ so sánh chi tiêu các danh mục của tôi tháng này.",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-chart", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

## Option D - test the transfer-form skill (trả nợ thẻ tín dụng)

Same wiring as Option C (`server.build_agent()`), just a different skill
(`skills/project/transfer-form/`) and a different `ui["type"]`
(`"transfer_form"` instead of `"chart"`).

This cell exercises **case (A)** only -- paying down the customer's own
credit card debt (`recipient`/`account_number` schema). See Option F below
for **case (B)**, transferring to a saved beneficiary via the real
`query_beneficiaries()` tool. The model verifies available credit itself
by reading `/customer_data.json` (`read_file`) -- there's no dedicated
tool for case (A), so expect only `read_file` calls in the output, no
tool call.

The question below asks about paying off a specific card's statement --
a direct match for the skill's description, similar to how Option C's
question echoes the chart skill's description.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason: avoids silently
# accumulating real history on GreenNode Memory across notebook re-runs.
transfer_agent = server.build_agent(checkpointer=InMemorySaver())

result = transfer_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tôi nên trả nợ thẻ Visa Signature bao nhiêu?",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-transfer-form", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

## Option E - test the spending-jar skill (tạo/chỉnh sửa hũ chi tiêu)

Same wiring as Option C/D (`server.build_agent()`), a third skill
(`skills/project/spending-jar/`) with two `ui["type"]` values:
`"create_jar"` (a new jar: `category_ids` and `initial_balance` -- the opening balance, a whole number >= 0 -- required, never a `jar_id`) and
`"edit_jar"` (an existing jar, identified by the `jar_id` the model copies from
`get_jar_summary`). Contract: `docs/spending-jar.md`.

The model reads the jars **live from the real `pfm` backend** with
`get_jar_summary()` (`GET /api/jar-summary`) and the customer's own categories
with `get_categories()` (`GET /api/categories`) -- so this cell needs network
access to the deployed `pfm`, like Option F -- and `/customer_data.json` for
`monthly_spending_summary` to ground the proposed amount. `server.py` drops the
`ui` when a `jar_id` or category is not one the model read in that turn
(`ui_guard.py`).

Two questions below, run in the same thread: the first should trigger
`create_jar`, the second `edit_jar` for a jar that already exists in pfm (its
`jar_id` comes from the summary, not from the conversation). The jar names are
live pfm data for `CIF_0001`; adjust question 2 if they have changed.


In [3]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason. This one matters
# even more here: the 2 questions below are DELIBERATELY run on the same
# thread (to test create_jar then edit_jar in one conversation) -- with the
# real checkpointer, re-running this cell keeps adding a 3rd, 5th, 7th...
# turn to the same real GreenNode thread each time, so "Tạo giúp tôi 1 hũ
# mua sắm" stops being a fresh request after the first run and starts
# correctly answering edit_jar instead (the model remembers offering it
# already) -- confusing if you don't expect it. InMemorySaver here means
# every notebook run is exactly the isolated 2-turn conversation it looks
# like in this cell.
jar_agent = server.build_agent(checkpointer=InMemorySaver())
JAR_THREAD = {"configurable": {"thread_id": "test-spending-jar", "actor_id": "CIF_0001"}}

for question in [
    "Tạo giúp tôi 1 hũ mua sắm hợp lý mỗi tháng.",  # expect create_jar
    "Tăng hạn mức hũ Ăn uống của tôi thêm 200 nghìn được không?",  # expect edit_jar (a jar that exists in pfm; jar_id comes from get_jar_summary)
]:
    result = jar_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=JAR_THREAD,
    )
    answer = extract_final_text(result["messages"][-1].content)
    answer, ui = extract_ui_block(answer)
    print(f"Q: {question}")
    print("ANSWER:\n", answer)
    print("\nUI:\n", ui)
    print("-" * 60)

/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/checkpoint/base/__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Model call: 6.24s
Calling tool: read_file
Calling tool: get_jar_summary
Calling tool: get_categories
Finished tool: read_file (0.01s)
Finished tool: get_jar_summary (0.26s)
Finished tool: get_categories (0.27s)
Model call: 11.17s
Q: Tạo giúp tôi 1 hũ mua sắm hợp lý mỗi tháng.
ANSWER:
 Danh mục Mua sắm hiện đang thuộc hũ **Hưởng thụ** (cùng với Giải trí).

Chi Mua sắm 3 tháng gần đây: 1.959.000đ (tháng 7) → 2.655.000đ (tháng 8) → 1.661.000đ (tháng 9), trung bình **~2.092.000đ/tháng**.

Mình đề xuất hũ **Mua sắm** riêng biệt hạn mức **2.200.000đ/tháng**, tách khỏi hũ Hưởng thụ để dễ kiểm soát.

Mình nạp sẵn **2.200.000đ** từ tiền chờ phân bổ vào hũ, bạn sửa được số này trên form rồi xác nhận nhé.

UI:
 {'type': 'create_jar', 'jar_name': 'Mua sắm', 'allocation_amount': 2200000, 'initial_balance': 2200000, 'category_ids': ['shopping'], 'reason': 'Chi Mua sắm 3 tháng gần nhất trung bình ~2.092.000đ/tháng; danh mục Mua sắm đang thuộc hũ Hưởng thụ sẽ chuyển sang hũ mới này, hũ Hưởng thụ còn p

## Option F - test the transfer-form skill (chuyển tiền cho người nhận đã lưu)

Same wiring as Option D, testing **case (B)** this time: transferring to a
saved beneficiary. Unlike Option D, this calls the tool
`query_beneficiaries()` (`tools/customer_data_tools.py`), which hits the
**real** `pfm` backend over HTTP (`GET /api/beneficiaries?cif=`, see
`backend_docs/pfm-read-api.md`) -- this cell needs network access to the
deployed `pfm` endpoint, and the beneficiary names below are real data for
`CIF_0001` at the time this was written (re-run `query_beneficiaries()`
directly if the seed data has since changed).

Expect the model to call `query_beneficiaries()` first, find "Lan" in the
result, then propose a `transfer_form` using the new
`beneficiary_id`/`category` schema (NOT `recipient`/`account_number` --
see `docs/transfer-form.md`).


In [4]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason.
beneficiary_transfer_agent = server.build_agent(checkpointer=InMemorySaver())

result = beneficiary_transfer_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Chuyển tiền cho Lan 500 nghìn với nội dung Thanh toán tiền ăn.",
            }
        ]
    },
    config={"configurable": {"thread_id": "test-transfer-form-beneficiary", "actor_id": "CIF_0001"}},
)

answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print("ANSWER:\n", answer)
print("\nUI:\n", ui)
assert ui is not None and ui["type"] == "transfer_form"
assert "beneficiary_id" in ui, "expected the pfm-lookup schema (beneficiary_id/category), not recipient/account_number"
assert ui["category"] in {
    "housing", "utilities", "subscriptions", "insurance", "dining",
    "transport", "shopping", "groceries", "entertainment", "health",
}


Model call: 5.76s
Calling tool: query_beneficiaries
Finished tool: query_beneficiaries (0.21s)
Model call: 3.48s
Calling tool: read_file
Finished tool: read_file (0.00s)
Model call: 6.77s
ANSWER:
 Mình đã điền sẵn form chuyển 500.000đ cho Nguyễn Thị Lan, bạn xem lại rồi bấm xác nhận nhé.

UI:
 {'type': 'transfer_form', 'beneficiary_id': 'b_stable_lan', 'amount': 500000, 'note': 'Thanh toan tien an', 'category': 'dining'}


## Option G - test the investment-guidance skill (short answer, decide-then-look-up)

Same wiring as Option D/F (`server.build_agent()`). Tests
`skills/project/investment-guidance/`: a goal-based investment question must get a
**4-5 sentence prose answer** (no tables/headings), grounded in the customer's real
income/spending/card debt, with any scenario folded into the conclusion ("Ngoài ra
bạn có thể cân nhắc ... nhưng cần khoảng ... vốn"). The flow is **decide first,
look up only when needed**: if the advice is "pay the card debt first" there is NO
MSB-product or news lookup; when a lookup is needed it is 1 search + 1 read, and
`middleware/tool_limits.py` hard-caps the calls per request. News is only used if
published within the last 4 days (older articles are refused by the tool); MSB pages
are fetched fresh (10-minute cache). Plain market-news questions are declined.

**Needs `TINYFISH_API_KEY` in `.env`** for the lookups.

Questions (fresh threads): plain news question -> declined; goal question with debt ->
no lookups; goal question as a customer with no jars; "which bank stock" -> at most
1 news search + 1 read.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
market_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", "Hôm nay chứng khoán thế nào, vì sao VN-Index giảm?"),  # plain news question -> declined/redirected
        ("CIF_0001", "Tôi muốn tăng tiền tiết kiệm thêm 10% đến cuối năm, tôi nên đầu tư gì?"),  # investment-guidance
        ("CIF_0001", "VN-Index giảm rồi, tôi nên mua cổ phiếu ngân hàng nào?"),  # no "buy X now"
        ("CIF_0002", "Tôi muốn tăng tiền tiết kiệm thêm 10% đến cuối năm, tôi nên đầu tư gì?"),  # own numbers
    ]
):
    result = market_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-market-{i}", "actor_id": actor_id}},
    )
    print(f"[{actor_id}] Q: {question}")
    print("ANSWER:\n", extract_final_text(result["messages"][-1].content))
    print("-" * 60)


## Option H - test MSB product answers (offline snapshot, with sources)

MSB product questions are answered from an OFFLINE snapshot of MSB's website and
text-layer PDFs (`data/msb_knowledge/`, built by `scripts/sync_msb_knowledge.py`), via ONE
local call to `search_msb_knowledge` (milliseconds, no network). Every passage carries its
source URL and fetch date, so the answer must end with a citation ("Nguồn: URL, dữ liệu lưu
ngày ..."). The old live web-lookup tools were removed and there is no separate skill: the rules live in `instructions.md`, so no extra model round-trip is spent reading a skill.

Expect: 1 tool call (plus the model round-trips), a 2-4 sentence answer, a source URL and
the snapshot date; deposit-rate questions say the rate table is not in the data and point to
MSB's rate page/app; refresh the snapshot with
`uv run python scripts/sync_msb_knowledge.py` (the tool warns when it is older than
`MSB_KB_MAX_AGE_DAYS`, default 7).


In [ ]:
import time

from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
msb_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", "MSB có thẻ tín dụng nào hoàn tiền không?"),
        ("CIF_0002", "Tiết kiệm Ong vàng của MSB là gì?"),
        ("CIF_0001", "Phí thường niên thẻ ghi nợ MSB là bao nhiêu?"),  # from the fee PDF
        ("CIF_0003", "Lãi suất tiết kiệm 12 tháng của MSB là bao nhiêu?"),  # rate table not in data -> link, no number
    ]
):
    started = time.time()
    result = msb_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-msb-{i}", "actor_id": actor_id}},
    )
    print(f"[{actor_id}] Q: {question}  ({time.time() - started:.1f}s)")
    print("ANSWER:\n", extract_final_text(result["messages"][-1].content))
    print("-" * 60)


## Option I - test the investment scenario projection (+ chart)

Tool `project_investment_growth` (`tools/investment_tools.py`, pure arithmetic)
computes, under a HYPOTHETICAL monthly return (default +2%/month -- a
product-owner assumption, not a measured market average): the capital needed to
earn a target gain by 31/12, and the value of a chosen capital at each
month-end for three scenarios (+r, flat 0%, -r). It also returns a ready-made
`ui-json` line chart. The agent must call it a "kịch bản giả định", always show
the flat and losing scenarios, and NOT assume the customer has a jar named
"tiết kiệm" -- capital comes from an amount the customer states, or is
estimated from balances (account - emergency fund - card debt; the money in
the jars is already part of the account balance, so it is not added), with 50% of the idle money proposed by default.

Answers are 4-5 sentences (charts only if the customer asks). Four questions, fresh threads: (1) target only, no stated amount; (2) a stated
amount + target; (3) another customer with no jars; (4) a large stated amount
(target reachable only in the +r scenario). Chart blocks are split out by
`extract_ui_block`, so `ui` prints the validated chart.


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver -- see Option C's comment, same reason.
scenario_agent = server.build_agent(checkpointer=InMemorySaver())

for i, (actor_id, question) in enumerate(
    [
        ("CIF_0001", " "),
        # ("CIF_0002", "Tôi có 60 triệu nhàn rỗi, muốn có thêm 10 triệu đến cuối năm, tôi nên đầu tư gì?"),
        # ("CIF_0003", "Tôi muốn có thêm 10 triệu đến cuối năm, tôi nên đầu tư gì?"),
        # ("CIF_0003", "Tôi có 400 triệu nhàn rỗi, muốn có thêm 10 triệu đến cuối năm, nên đầu tư thế nào?"),
    ]
):
    result = scenario_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"configurable": {"thread_id": f"test-scenario-{i}", "actor_id": actor_id}},
    )
    answer, ui = extract_ui_block(extract_final_text(result["messages"][-1].content))
    print(f"[{actor_id}] Q: {question}")
    print("ANSWER:\n", answer)
    print("UI (chart):\n", ui)
    print("-" * 60)


Model call: 6.63s
Calling tool: read_file
Finished tool: read_file (0.00s)
Model call: 5.26s
Calling tool: search_msb_products
Calling tool: search_msb_products
Calling tool: search_msb_products
Finished tool: search_msb_products (1.23s)
Finished tool: search_msb_products (1.57s)
Finished tool: search_msb_products (1.87s)
Model call: 6.48s
Calling tool: search_msb_products
Calling tool: read_msb_product_page
Calling tool: read_msb_product_page
Finished tool: search_msb_products (1.25s)
Finished tool: read_msb_product_page (3.96s)
Finished tool: read_msb_product_page (4.82s)
Model call: 5.11s
Calling tool: search_msb_products
Calling tool: search_msb_products
Finished tool: search_msb_products (1.26s)
Finished tool: search_msb_products (1.26s)
Model call: 5.27s
Calling tool: read_msb_product_page
Finished tool: read_msb_product_page (2.82s)
Model call: 20.76s
[CIF_0001] Q: MSB Có sản phẩm gì để đầu tư không
ANSWER:
 MSB có 3 nhóm sản phẩm để bạn đầu tư/sinh lời:

**1. Tiết kiệm & tiền g

## Option J - test `POST /jar-distribute` and `POST /jar-rebalance` (không cần chat)

The two endpoints for screens that already know what to do, so there is no conversation
(`jar_endpoint.py`; contract, error codes and what the agent is handed in `docs/jars/endpoints.md`).
Called in-process with `TestClient(server.app)` -- the same wiring `/chat` lives in:

- **`POST /jar-distribute`** `{"user_id", "amount"}`: share an amount of the customer's **unallocated money** into the jars'
  balances (never the limits) -> a `distribute_amount` `ui`. The agent decides which jars and how much.
- **`POST /jar-rebalance`** `{"user_id", "from_jar_id", "amount"}`: move an amount of one jar's balance to one or more other
  jars -> a `rebalance_jars` `ui` (`from_jar_id` + `allocations`).

For both, the server reads the jar summary **and the last 3 months of spending per jar** and hands them to the
model, so the split follows real need. The cell reads the real `jar-summary` from `pfm` first (so it picks a real jar
and survives data changes), calls each endpoint, then runs the SAME deterministic checks the server uses
(`jar_checks.check_distribute` / `check_rebalance`) and asserts no hard rule is broken. It needs network access to `pfm`
and to the model, and `SERVER_API_KEY` in `../.env` (the endpoints use the same `X-API-Key` as `/chat`). Each call takes
~10-40 seconds. When the pool or the jar cannot cover the amount the server answers by itself with `ui: null` -- that is a
valid result too, the cell prints it.

In [5]:
import os

from fastapi.testclient import TestClient

import server
from jar_checks import check_distribute, check_rebalance, pool_available, source_available
from tools.customer_data_tools import fetch_jar_summary

assert os.environ.get("SERVER_API_KEY"), f"Set SERVER_API_KEY in {REPO_ROOT / '.env'} first"

client = TestClient(server.app)
HEADERS = {"X-API-Key": os.environ["SERVER_API_KEY"]}
CUSTOMER = "CIF_0001"

# The real jar-summary decides which jar to use, so this cell keeps working when the pfm data changes.
summary = fetch_jar_summary(CUSTOMER)
print("jars (limit, balance):", {j["label"]: (j["limit"], j["balance"]) for j in summary["jars"]})
print("unallocated:", summary["unallocated"])
print("=" * 60)

# --- POST /jar-distribute: share unallocated money into the jars (the agent picks the jars) ---
AMOUNT = 500_000
response = client.post("/jar-distribute", headers=HEADERS, json={"user_id": CUSTOMER, "amount": AMOUNT})
assert response.status_code == 200, response.text
body = response.json()
print(f"DISTRIBUTE {AMOUNT} (unallocated now: {pool_available(summary)})")
print("ANSWER:\n", body["answer"])
print("\nUI:\n", body["ui"])
if body["ui"] is not None:
    verdict = check_distribute(summary, AMOUNT, body["ui"])
    assert verdict.hard == [], verdict.hard
    print("soft notes (only wording / money left undistributed):", verdict.soft)
print("=" * 60)

# --- POST /jar-rebalance: move balance from the jar that holds the most into other jars ---
giver = max((j for j in summary["jars"] if j.get("balance") is not None), key=lambda j: j["spendable"] or 0, default=None)
if giver is None:
    print("No jar has a balance yet, so there is nothing to move -- skipping /jar-rebalance.")
else:
    amount = min(500_000, source_available(summary, giver["id"]))
    response = client.post(
        "/jar-rebalance",
        headers=HEADERS,
        json={"user_id": CUSTOMER, "from_jar_id": giver["id"], "amount": amount},
    )
    assert response.status_code == 200, response.text
    body = response.json()
    print(f"REBALANCE {amount} from {giver['label']} (spendable {giver['spendable']})")
    print("ANSWER:\n", body["answer"])
    print("\nUI:\n", body["ui"])
    if body["ui"] is not None:
        verdict = check_rebalance(summary, giver["id"], amount, body["ui"])
        assert verdict.hard == [], verdict.hard
        print("soft notes (only wording / money left in the giver):", verdict.soft)

jars (budgetLimit, remaining): {'Thiết yếu': (8000000, 51000), 'Ăn uống': (4000000, 1813000), 'Di chuyển': (1500000, 796000), 'Hưởng thụ': (2500000, 1296000), 'Sức khỏe': (1000000, 1000000), 'Tiết kiệm': (None, None)}
unallocated: 13044000


distribute-amount: attempt 1 not clean: hard=[] soft=['reason không được nhắc tên nội bộ (shortfall); dùng cách nói thường của khách.']
distribute-amount: model call failed (attempt 2)
Traceback (most recent call last):
  File "/media/daonguyen/Dual/MSB/MSBHackathon/personal-card-agent/jar_endpoint.py", line 170, in _run
    draft = _propose(task.prompt, task.schema, messages)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/daonguyen/Dual/MSB/MSBHackathon/personal-card-agent/jar_endpoint.py", line 143, in _propose
    return agent.invoke({"messages": messages})["structured_response"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/main.py", line 3365, in invoke
    for chunk in self.stream(
                 ^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/main.py", line 2759, in stream
    for _ in runner.tick(
           

COVER Thiết yếu: spend 1051000, shortfall 1000000
ANSWER:
 Hũ Thiết yếu đang thiếu 1.000.000đ cho khoản chi 1.051.000đ.

Đề xuất lấy đủ 1.000.000đ từ tiền chưa phân bổ — không cần động đến hũ nào khác.

UI:
 {'type': 'rebalance_jars', 'target_jar_id': 'essentials', 'shortfall': 1000000, 'moves': [{'from_jar_id': 'pool', 'amount': 1000000}], 'reason': 'Tiền chưa phân bổ (13.044.000đ) đủ để bù trọn 1.000.000đ shortfall của hũ Thiết yếu, nên không cần lấy từ hũ khác.'}
soft notes (only the preferred order/wording): []


distribute-amount: model call failed (attempt 1)
Traceback (most recent call last):
  File "/media/daonguyen/Dual/MSB/MSBHackathon/personal-card-agent/jar_endpoint.py", line 170, in _run
    draft = _propose(task.prompt, task.schema, messages)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/media/daonguyen/Dual/MSB/MSBHackathon/personal-card-agent/jar_endpoint.py", line 143, in _propose
    return agent.invoke({"messages": messages})["structured_response"]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/main.py", line 3365, in invoke
    for chunk in self.stream(
                 ^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/main.py", line 2759, in stream
    for _ in runner.tick(
             ^^^^^^^^^^^^
  File "/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/langgraph/pregel/_runner.py", line 167, in tick
 

AssertionError: {"detail":"The model is unavailable, try again shortly"}

## Option K - test the jar flows in chat (`POST /chat`)

The chat side of the jar contract (`docs/jars/`), run
through the real `POST /chat` route with `TestClient(server.app)` on an in-memory agent
(`server.agent` is swapped for `server.build_agent(checkpointer=InMemorySaver())`, so every run
starts a fresh thread -- see Option C). Going through the route means the shape validators
(`text_utils.py`) **and** `ui_guard.py` run, exactly as they do for the frontend. The model
reads the live jars from `pfm` (`get_jar_summary`, `get_categories`), so the cell needs network
access to `pfm` and to the model, and the questions are built from the real jars (they keep working
when the data changes). Four calls, ~15-40 seconds each:

1. **Category move** -- "chuyển danh mục X từ hũ A sang hũ B". Expect ONE `edit_jar` for the
   receiving jar B with B's full category list plus X (an `edit_jar` for A would leave X in no jar).
2. **Put unallocated money into a jar** (skill `distribute-amount`) -- "chuyển 300 nghìn vào hũ X". Expect
   `distribute_amount` with that jar and the amount, and the SAME deterministic checks the server's endpoint uses
   (`jar_checks.check_distribute`) must find no hard error. The money comes only from the unallocated pool
   (`unallocated`), never from another jar; when the pool is empty the model should say so and return no `ui`.
3. **A jar that only exists in the conversation** -- ask for a new jar "Liên thiên", then ask to
   raise it in the next turn. The first turn may propose `create_jar`, but the customer has not
   confirmed it, so the jar is not in `get_jar_summary`: expect **no `ui`** on the second turn (the
   model must not invent a `jar_id`, and `ui_guard.py` drops the form if it does).


In [4]:
import os

import httpx
from fastapi.testclient import TestClient
from langgraph.checkpoint.memory import InMemorySaver

import server
from jar_checks import check_distribute, pool_available
from tools.customer_data_tools import PFM_BASE_URL, fetch_jar_summary

assert os.environ.get("SERVER_API_KEY"), f"Set SERVER_API_KEY in {REPO_ROOT / '.env'} first"

# In-memory agent: every run of this cell is a fresh thread (see Option C for why).
server.agent = server.build_agent(checkpointer=InMemorySaver())
client = TestClient(server.app)
HEADERS = {"X-API-Key": os.environ["SERVER_API_KEY"]}
CUSTOMER = "CIF_0001"


def chat(message: str, thread: str) -> dict:
    response = client.post("/chat", headers=HEADERS, json={"message": message, "user_id": CUSTOMER, "thread_id": thread})
    assert response.status_code == 200, response.text
    body = response.json()
    print(f"Q: {message}\nANSWER:\n {body['answer']}\n\nUI:\n {body['ui']}\n" + "-" * 60)
    return body


def vnd(amount: int) -> str:
    return f"{amount:,}".replace(",", ".") + "đ"


# The real jars decide the questions, so this cell keeps working when the pfm data changes.
summary = fetch_jar_summary(CUSTOMER)
categories = httpx.get(f"{PFM_BASE_URL}/api/categories", params={"cif": CUSTOMER}, timeout=10).json()
with_limit = [j for j in summary["jars"] if j.get("limit") is not None]

# --- 1. move a category from jar A to jar B: ONE edit_jar, for the RECEIVING jar B ---
receiver = with_limit[0]
giver = next(j for j in summary["jars"] if j["id"] != receiver["id"] and j["categoryIds"])
moved = giver["categoryIds"][0]
moved_label = next(c["label"] for c in categories if c["id"] == moved)
ui = chat(f"Chuyển danh mục {moved_label} từ hũ {giver['label']} sang hũ {receiver['label']}.", "k-move")["ui"]
assert ui is not None and ui["type"] == "edit_jar", "expected an edit_jar"
assert ui["jar_id"] == receiver["id"], "must edit the RECEIVING jar, not the giving one"
assert set(ui["category_ids"]) == set(receiver["categoryIds"]) | {moved}, "the receiving jar's full list + the moved category"

# --- 2. put unallocated money into a jar (skill distribute-amount): same contract and same checks as POST /jar-distribute ---
target = with_limit[0]
amount = min(300_000, pool_available(summary)) or 300_000
ui = chat(f"Chuyển {vnd(amount)} vào hũ {target['label']} giúp tôi.", "k-topup")["ui"]
if ui is None:
    print("(no ui: the unallocated money is not enough, or the model explained instead -- valid)")
else:
    assert ui["type"] == "distribute_amount", ui["type"]
    verdict = check_distribute(summary, amount, ui)
    assert verdict.hard == [], verdict.hard
    print("soft notes (only wording / leaving some undistributed):", verdict.soft)

# --- 3. a jar that only exists in the conversation cannot be edited: no ui ---
chat("Tạo giúp tôi hũ Liên thiên 500 nghìn.", "k-ghost")  # may propose create_jar; the customer never confirms it
ui = chat("Tăng hũ Liên thiên lên 700 nghìn giúp tôi.", "k-ghost")["ui"]
assert ui is None, "an edit_jar for a jar that is not in get_jar_summary must never reach the frontend"


/home/daonguyen/anaconda3/envs/msb/lib/python3.12/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Model call: 7.95s
Calling tool: read_file
Calling tool: get_jar_summary
Finished tool: read_file (0.00s)
Finished tool: get_jar_summary (0.21s)
Model call: 6.15s
Calling tool: get_categories
Finished tool: get_categories (0.22s)
Model call: 10.78s
Q: Chuyển danh mục Ăn uống từ hũ Ăn uống sang hũ Thiết yếu.
ANSWER:
 Danh mục Ăn uống sẽ chuyển từ hũ Ăn uống sang hũ Thiết yếu.

Hũ Ăn uống còn phụ trách Nhu yếu phẩm, hạn mức hai hũ giữ nguyên, bạn xem lại rồi xác nhận nhé.

UI:
 {'type': 'edit_jar', 'jar_id': 'essentials', 'jar_name': 'Thiết yếu', 'allocation_amount': 9600000, 'category_ids': ['housing', 'utilities', 'insurance', 'subscriptions', 'dining'], 'reason': 'Chuyển danh mục Ăn uống từ hũ Ăn uống sang hũ Thiết yếu; hũ Ăn uống còn phụ trách Nhu yếu phẩm, hạn mức hai hũ giữ nguyên'}
------------------------------------------------------------
Model call: 3.83s
Calling tool: get_jar_summary
Calling tool: read_file
Finished tool: read_file (0.00s)
Finished tool: get_jar_summary (0.22s

## Option L - test the `clarify-options` skill (`clarify_options`)

`clarify_options` (`skills/project/clarify-options/SKILL.md`, contract in
`docs/clarify-options.md`) is a cross-cutting widget: instead of an open-ended
follow-up question, the agent offers 1-4 concrete choices as buttons for a
question it genuinely cannot decide on its own. It is a **last resort** --
so it needs a real ambiguity to trigger, not a stubbed one.

`CIF_0001` has 6 real jars (Thiet yeu/An uong/Di chuyen/Huong thu/Suc
khoe/Tiet kiem) -- more than the schema's 4-option cap, so the skill's own
"KHONG DUOC liet ke qua 4 phuong an" rule forbids offering all 6 as buttons
for a fully unscoped "which jar" question (that must fall back to text --
not a bug, that is what an earlier run of this cell showed). To get a real,
non-stubbed jar-selection `clarify_options` reliably, this cell states the
amount up front (so that is not part of the ambiguity -- see below) and
names the jar with a description that does not literally match any of the
6 real labels ("hu chi tieu hang ngay"). The agent cannot resolve that
against the real jars it just read, so it must ask **which real jar** the
customer meant -- offering up to 4 of them (real names + real
`budgetLimit`) as buttons, per the schema's own cap.

Same wiring as Options C-I (`server.build_agent()` + `agent.invoke()`
directly, `InMemorySaver` -- see Option C's comment for why), not the HTTP
endpoint style of Options J/K. Two turns, same thread:
1. Ask "Tang han muc hu chi tieu hang ngay cua toi len 300.000d giup toi."
   -- amount is already stated (300.000d), so the only unstated thing is
   *which* real jar. Expect `ui["type"] == "clarify_options"` with **one**
   question and up to 4 real jars (name + budgetLimit) as options.
2. Simulate the customer tapping the first option -- send its exact string
   back as the next chat message (the whole reply mechanism for a single
   question: no structured payload, just the button's text as a normal
   message). Expect the agent to now propose `edit_jar` for that jar, with
   its limit raised by 300.000d, having re-read `get_jar_summary` this turn.

**Verified with 5/5 real runs** against `CIF_0001`'s live data: every run
produced a clean single-question `clarify_options` with exactly 4 real jars
as options (the model drops 2 of the 6 each time -- which 2 varies, but the
cap and the real names/amounts held every time). Still printed rather than
asserted, since the model is non-deterministic and could occasionally do
something else (e.g. ask by plain text instead) -- if that happens, rerun
the cell.


In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason.
clarify_agent = server.build_agent(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "test-clarify-options", "actor_id": "CIF_0001"}}

# Amount is stated (300.000d) so it is not part of the ambiguity. "hu chi tieu hang ngay" matches
# none of CIF_0001's 6 real jars (Thiet yeu/An uong/Di chuyen/Huong thu/Suc khoe/Tiet kiem) -- the
# agent must ask which real one was meant, capped at 4 options per the schema.
question = "Hỏi tôi các câu hỏi để biết nhu cầu đầu tư của tôi."
result = clarify_agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print(f"Q: {question}")
print("ANSWER:\n", answer)
print("\nUI:\n", ui)
print("-" * 60)

if ui is not None and ui.get("type") == "clarify_options":
    # Only one question this time -- tap its first button, send that exact string as the next message.
    pick = ui["questions"][0]["options"][0]
    result2 = clarify_agent.invoke({"messages": [{"role": "user", "content": pick}]}, config=config)
    answer2 = extract_final_text(result2["messages"][-1].content)
    answer2, ui2 = extract_ui_block(answer2)
    print(f"Q (đã chọn): {pick}")
    print("ANSWER:\n", answer2)
    print("\nUI:\n", ui2)
else:
    print("Lần này model không dùng clarify_options (không tất định, xem ghi chú ở trên) -- chạy lại ô này để thử lại.")


Model call: 7.48s
Calling tool: read_file
Finished tool: read_file (0.00s)
Model call: 9.24s
Q: Hỏi tôi các câu hỏi để biết nhu cầu đầu tư của tôi.
ANSWER:
 Để tư vấn đầu tư phù hợp, tôi cần hiểu thêm về bạn. Tôi sẽ hỏi từng bước nhé.

**Bạn đang có số tiền nào dự định đầu tư?** (ví dụ 30 triệu, 60 triệu...)

**Mục tiêu của bạn khi đầu tư là gì?** — tích lũy dài hạn, sinh thêm thu nhập hàng tháng, hay đạt một số tiền cụ thể vào thời điểm nào?

**Bạn có thể chấp nhận rủi ro ở mức nào?** — không muốn mất vốn, chấp nhận biến động nhẹ, hay sẵn sàng rủi ro cao để kỳ vọng lợi nhuận lớn hơn?

**Bạn dự kiến giữ khoản đầu tư trong bao lâu?** — dưới 6 tháng, 1–2 năm, hay trên 3 năm?

Bạn trả lời những câu trên, tôi sẽ tư vấn dựa trên tình hình tài chính thật của bạn.

UI:
 None
------------------------------------------------------------
Lần này model không dùng clarify_options (không tất định, xem ghi chú ở trên) -- chạy lại ô này để thử lại.


## Option M - test putting an amount into a jar (`distribute_amount`, agent called directly)

"Chuyển 300 nghìn vào hũ Tiết kiệm" puts a specific amount of **unallocated money** (tiền chưa phân bổ, "Chờ phân bổ") into a jar's balance -- the job of
the `distribute_amount` form (skill `skills/project/distribute-amount/`, contract `docs/jars/distribute-amount.md`).
The money comes only from the unallocated pool (`unallocated`) and is **never taken from another jar** (không lấy từ hũ khác). Several jars in one sentence
("chia 1 triệu vào Ăn uống và Di chuyển") go into ONE form. A jar with no balance yet (like Tiết kiệm) can still receive: the deposit is
its first balance. The agent only **proposes**: nothing is moved here -- `pfm` writes it when the customer confirms.

Same wiring as Options C-I and L: `server.build_agent()` + `agent.invoke()` directly, `InMemorySaver` (see Option C's comment). No
`TestClient`, no `POST /chat`, no `POST /jar-distribute`, so this is the model + skill + tools only. It reads the real jars from `pfm`
(`get_jar_summary`), so it needs network access to `pfm` and the model.

The cell prints the jar and the unallocated money BEFORE asking: if the pool holds less than the amount, the right answer is **no `ui`**
and a sentence with the real figure. When a form comes back, the cell runs the same deterministic check the endpoint uses
(`jar_checks.check_distribute`): `hard` must be `[]`; `soft` is only wording or money left undistributed.

Not asserted: the model is non-deterministic, so the cell prints what happened instead of failing on an unlucky run.

**Known gap:** `pfm`'s frontend has no card for `distribute_amount` yet (its `agent-api.ts` knows chart, transfer_form, create_jar, edit_jar,
rebalance_jars, clarify_options), so a real UI would show only `answer` until `pfm` adds it.


In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from jar_checks import check_distribute
from text_utils import extract_final_text, extract_ui_block
from tools.customer_data_tools import fetch_jar_summary

# InMemorySaver -- see Option C's comment, same reason.
topup_agent = server.build_agent(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "test-jar-topup", "actor_id": "CIF_0001"}}

# What the jar and the pool really hold right now (the same live pfm data the agent reads). The money comes from `unallocated`;
# `balance: None` = the jar has no balance yet, which does not stop it from receiving.
summary = fetch_jar_summary("CIF_0001")
jar = next((j for j in summary["jars"] if j["label"] == "Tiết kiệm"), None)
print("Tiết kiệm ->", None if jar is None else {k: jar.get(k) for k in ("limit", "balance", "spendable")})
print("unallocated:", summary["unallocated"])
print("=" * 60)

question = "Chuyển 300 nghìn vào hũ Tiết kiệm giúp tôi."
result = topup_agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print(f"Q: {question}")
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

if ui is not None and ui.get("type") == "distribute_amount":
    verdict = check_distribute(summary, 300_000, ui)
    print("\ncheck_distribute -- hard errors (must be []):", verdict.hard)
    print("check_distribute -- soft notes:", verdict.soft)
else:
    print("\nKhông có distribute_amount lần này -- hợp lệ nếu tiền chưa phân bổ không đủ 300.000đ (xem unallocated ở trên).")


## Option N - test moving money from one jar to others (`rebalance_jars`, agent called directly)

"Chuyển 500 nghìn từ hũ Ăn uống sang hũ Di chuyển" moves balance **out of a jar** into one or more other jars -- the job of the
`rebalance_jars` form (skill `skills/project/rebalance-jars/`, contract `docs/jars/rebalance-jars.md`): ONE giving jar
(`from_jar_id`) and one or more receivers (`allocations`, each `to_jar_id` + `amount`); the total can be at most what the giver
holds (`spendable`) and the giver never receives. The unallocated pool is not involved here -- that is Option M
(`distribute-amount`). The agent only **proposes**: nothing is moved -- `pfm` writes it when the customer confirms.

Same wiring as Option M: `server.build_agent()` + `agent.invoke()` directly, `InMemorySaver`. No `TestClient`, no `POST /chat`.
The question is built from the real jars (`fetch_jar_summary`): the jar that holds the most gives to another jar, so the cell
survives data changes. When a form comes back, the cell runs `jar_checks.check_rebalance`: `hard` must be `[]`.

Not asserted: the model is non-deterministic, so the cell prints what happened instead of failing on an unlucky run.

**Known gap:** `pfm`'s `AgentRebalanceCard` still expects the old shape (`target_jar_id`/`shortfall`/`moves`); it has to move to `from_jar_id` +
`allocations`.

In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from jar_checks import check_rebalance
from text_utils import extract_final_text, extract_ui_block
from tools.customer_data_tools import fetch_jar_summary

# InMemorySaver -- see Option C's comment, same reason.
rebalance_agent = server.build_agent(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "test-jar-rebalance", "actor_id": "CIF_0001"}}

# The jar that holds the most gives to another jar; the money comes from the giver, never from `unallocated`.
summary = fetch_jar_summary("CIF_0001")
funded = [j for j in summary["jars"] if j.get("balance") is not None]
giver = max(funded, key=lambda j: j["spendable"] or 0)
receiver = next(j for j in summary["jars"] if j["id"] != giver["id"])
print("giver   ->", {k: giver.get(k) for k in ("label", "limit", "balance", "spendable")})
print("receiver ->", {k: receiver.get(k) for k in ("label", "limit", "balance", "spendable")})
print("=" * 60)

AMOUNT = min(500_000, giver["spendable"] or 0)
question = f"Chuyển {AMOUNT:,} từ hũ {giver['label']} sang hũ {receiver['label']} giúp tôi.".replace(",", ".")
result = rebalance_agent.invoke({"messages": [{"role": "user", "content": question}]}, config=config)
answer = extract_final_text(result["messages"][-1].content)
answer, ui = extract_ui_block(answer)
print(f"Q: {question}")
print("ANSWER:\n", answer)
print("\nUI:\n", ui)

if ui is not None and ui.get("type") == "rebalance_jars":
    verdict = check_rebalance(summary, giver["id"], AMOUNT, ui)
    print("\ncheck_rebalance -- hard errors (must be []):", verdict.hard)
    print("check_rebalance -- soft notes:", verdict.soft)
else:
    print("\nKhông có rebalance_jars lần này -- hợp lệ nếu hũ cho không đủ số dư (xem giver ở trên).")

## Option O - test the user-preferences middleware ("Hướng dẫn cho M-You")

Writes a preference row for `CIF_0001` straight to `pfm`'s real `PUT /api/agent-preferences`
(`src/app/api/agent-preferences/route.ts` in the `pfm` repo) -- the Agent repo has no tool/endpoint
for *writing* this (by design: the Agent only ever **reads** it, see
`middleware/seed_user_preferences.py`, todo.md 1.45). This cell stands in for the customer filling in
the "Hướng dẫn cho M-You" sheet on `pfm` (gear icon next to the delete-history button) -- it needs
network access to the deployed `pfm` instance, same as Option F/K.

After writing, asks the agent (`server.build_agent()`, a fresh `InMemorySaver` thread, same reason as
Option C) what it knows about the customer. A correct answer should: address the customer as
`addressAs`, keep roughly the requested `tone`/`responseLength`, and weave in `about` as something the
customer mentioned about themselves -- without treating any of it as a verified financial fact or as
an instruction that can skip auth/OTP (see the preamble `seed_user_preferences.py` always prepends).

In [ ]:
import httpx

from tools.customer_data_tools import PFM_BASE_URL

# Mirrors what the customer would type into pfm's "Hướng dẫn cho M-You" sheet. `PUT` upserts the WHOLE
# row (pfm has no partial-patch for this) -- see src/lib/agent-preferences-store.ts in the pfm repo.
preferences_cif_0001 = {
    "cif": "CIF_0001",
    "responseLength": "concise",
    "tone": "playful",
    "riskAppetite": "aggressive",
    "addressAs": "chị Lan",
    "about": "Tôi thích bóng đá và thường xuyên đầu tư cổ phiếu.",
    "notes": "Trả lời ngắn gọn, đi thẳng vào vấn đề, không cần rào trước đón sau.",
}

put_resp = httpx.put(f"{PFM_BASE_URL}/api/agent-preferences", json=preferences_cif_0001, timeout=10.0)
put_resp.raise_for_status()
print("PUT ->", put_resp.json())

# Read back exactly the way fetch_user_preferences() / the middleware will.
get_resp = httpx.get(f"{PFM_BASE_URL}/api/agent-preferences", params={"cif": "CIF_0001"}, timeout=10.0)
get_resp.raise_for_status()
print("GET ->", get_resp.json())
assert get_resp.json() == {k: v for k, v in preferences_cif_0001.items() if k != "cif"}

In [ ]:
import server
from langgraph.checkpoint.memory import InMemorySaver
from text_utils import extract_final_text, extract_ui_block

# InMemorySaver -- see Option C's comment, same reason: a fresh thread so only the preferences just
# written above (via SeedUserPreferences' before_agent, read live from pfm) shape the answer, not any
# leftover conversation history.
preferences_agent = server.build_agent(checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "test-user-preferences", "actor_id": "CIF_0001"}}

result = preferences_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Bạn biết gì về tôi, và bạn sẽ trả lời tôi theo phong cách nào?",
            }
        ]
    },
    config=config,
)
print(extract_final_text(result["messages"]))
print("ui:", extract_ui_block(result["messages"]))

In [ ]:
# Cleanup: clear CIF_0001's preferences back to empty so this shared test persona doesn't keep
# "chị Lan / thích bóng đá" standing instructions for every other notebook cell/person after this one.
# Comment this out if you want to keep inspecting the preferences a bit longer first.
empty_preferences = {"cif": "CIF_0001", "responseLength": None, "tone": None, "riskAppetite": None,
                      "addressAs": "", "about": "", "notes": ""}
reset_resp = httpx.put(f"{PFM_BASE_URL}/api/agent-preferences", json=empty_preferences, timeout=10.0)
reset_resp.raise_for_status()
print("reset ->", reset_resp.json())

## Option P - test the goal skills: create a goal and add an asset (`create_goal`, `add_asset`)

Skills `goal-manage` and `goal-asset`, run through the real `POST /chat` with `TestClient(server.app)` on an in-memory agent
(same wiring as Option K: validators in `text_utils.py` **and** `ui_guard.py` run). The model reads the customer's goals live from `pfm`
(`get_goals`, `get_goal_market`: today by pfm's own clock, main balance, monthly cash flow, each goal's monthly need and auto-debit,
assets), so the cell needs network access to `pfm` and to the model. **To try against your local pfm** set `PFM_URL = "http://localhost:3000"`
in the setup cell below (needs `npm run dev` in `pfm`); by default it reads the deployed server. Each call takes ~15-40 s.

Run the setup cell once, then any case cell on its own (the model is not deterministic: re-run a case that fails once before concluding).

| # | Case | Expect |
|---|---|---|
| 1 | "Tôi muốn mua xe 600 triệu, cuối năm 2027, nạp trước 5 triệu" | `create_goal` with a monthly plan (`monthly_amount` + `monthly_day`) |
| 2 | "Đặt mục tiêu du lịch 30 triệu trong 6 tháng" | `create_goal`, date counted from pfm's `today` |
| 3 | "Tôi muốn đặt mục tiêu mua nhà" | `clarify_options` (skill `clarify-options`) with 2 questions: the amount and the time frame; clicking one answer each then gives `create_goal` |
| 4-6 | add a hand-valued asset / a stock / a bank deposit to an existing goal | `add_asset` with the right fields for each group |
| 7 | "mỗi tháng cần góp bao nhiêu" | an answer with the real numbers from `get_goals` |
| 7b | all 9 asset categories one by one | each one with its own form (no `value` for market and interest assets) |

In [2]:
import os

assert os.environ.get("SERVER_API_KEY"), f"Set SERVER_API_KEY in {REPO_ROOT / '.env'} first"

import httpx
from fastapi.testclient import TestClient
from langgraph.checkpoint.memory import InMemorySaver

import server
import tools.customer_data_tools as customer_tools
import tools.goal_tools as goal_tools
from text_utils import is_valid_ui
from tools.goal_tools import fetch_goals_snapshot, months_left, render_goals, summarize_goals

# --- pfm dùng để ĐỌC (và để ghi ở ô 8). Mặc định là server đã deploy; muốn thử trên máy: PFM_URL = "http://localhost:3000" ---
PFM_URL = None
if PFM_URL:
    customer_tools.PFM_BASE_URL = goal_tools.PFM_BASE_URL = PFM_URL
PFM = goal_tools.PFM_BASE_URL
print("pfm:", PFM)

# Agent trong bộ nhớ: mỗi lần chạy là một thread mới. TestClient đi qua route thật nên validator và ui_guard đều chạy.
server.agent = server.build_agent(checkpointer=InMemorySaver())
client = TestClient(server.app)
HEADERS = {"X-API-Key": os.environ["SERVER_API_KEY"]}
CUSTOMER = "CIF_0001"


def vnd(amount) -> str:
    return f"{int(amount):,}".replace(",", ".") + "đ"


def chat(message: str, thread: str) -> dict:
    response = client.post("/chat", headers=HEADERS, json={"message": message, "user_id": CUSTOMER, "thread_id": thread})
    assert response.status_code == 200, response.text
    body = response.json()
    print(f"Q: {message}\nANSWER:\n{body['answer']}\n\nUI:\n{body['ui']}\n" + "-" * 60)
    return body


def expected_monthly(ui: dict) -> int:
    """Số góp mỗi tháng theo công thức của skill: (mục tiêu - nạp ban đầu) / số tháng tới hạn, làm tròn lên nghìn."""
    import math

    gap = max(0, ui["target_amount"] - ui["initial_deposit"])
    months = months_left(goals["today"], ui["target_date"])
    return gap if months == 0 else int(math.ceil(gap / months / 1000) * 1000)


def pfm_actions(ui: dict) -> list[dict]:
    """Điều pfm sẽ ghi khi khách bấm xác nhận (giống các thẻ trong giao diện)."""
    if ui["type"] == "create_goal":
        plan = {"standingOrder": {"amount": ui["monthly_amount"], "dayOfMonth": ui["monthly_day"]}} if ui.get("monthly_amount") else {}
        return [{"type": "createGoal", "input": {"name": ui["goal_name"], "template": ui["template"], "targetAmount": ui["target_amount"],
                                                 "targetDate": ui["target_date"], "flexibility": "date_flexible", "initialDeposit": ui["initial_deposit"], **plan}}]
    if ui["type"] == "add_asset":
        base = {"type": "addAsset", "goalId": ui["goal_id"], "name": ui["asset_name"], "category": ui["category"], "countPercent": ui["count_percent"]}
        if ui.get("institution"):
            base["institution"] = ui["institution"]
        if "instrument_id" in ui:
            return [{**base, "value": 0, "source": "estimated", "instrumentId": ui["instrument_id"], "quantity": ui["quantity"]}]
        if "accrual" in ui:
            a = ui["accrual"]
            accrual = {"principal": a["principal"], "ratePa": a["rate_pa"], "startDate": a["start_date"], **({"maturityDate": a["maturity_date"]} if "maturity_date" in a else {})}
            return [{**base, "value": a["principal"], "source": "estimated", "accrual": accrual}]
        return [{**base, "value": ui["value"], "source": "self_reported"}]
    if ui["type"] == "edit_goal":
        out = []
        if "new_target_date" in ui:
            out.append({"type": "extend", "goalId": ui["goal_id"], "newTargetDate": ui["new_target_date"]})
        if "new_target_amount" in ui:
            out.append({"type": "reduceTarget", "goalId": ui["goal_id"], "newTargetAmount": ui["new_target_amount"]})
        if "monthly_amount" in ui:
            out.append({"type": "setStandingOrder", "goalId": ui["goal_id"], "amount": ui["monthly_amount"], "dayOfMonth": ui["monthly_day"]})
        return out
    return []


def show_ui(ui: dict | None, expect: str) -> None:
    assert ui is not None, "agent không trả form"
    assert ui["type"] == expect, f"cần {expect}, nhận {ui['type']}"
    assert is_valid_ui(ui), "form sai format"
    print("pfm sẽ ghi khi khách xác nhận:")
    for action in pfm_actions(ui):
        print("  ", action)


# Dữ liệu thật quyết định câu hỏi, nên notebook vẫn chạy khi dữ liệu pfm đổi.
goals = summarize_goals(fetch_goals_snapshot(CUSTOMER))
print(render_goals(goals))
open_goals = [g for g in goals["goals"] if g["status"] not in {"closed", "completed"}]
TARGET_GOAL = open_goals[0] if open_goals else None
print("\nMục tiêu dùng cho các ô thêm tài sản:", TARGET_GOAL["name"] if TARGET_GOAL else "(chưa có mục tiêu nào: chạy ô 1 và 8 trước)")

pfm: https://endpoint-6529c773-a2b3-4a42-b28a-6bcdced3ceff.agentbase-runtime.aiplatform.vngcloud.vn
Hôm nay (đồng hồ của hệ thống): 2026-10-21. Số dư tài khoản chính: 21.726.333đ.
Thu chi hằng tháng của khách: thu nhập 31.433.333đ, chi tiêu 15.093.333đ, trả nợ có sẵn 3.500.000đ, còn dư 12.840.000đ; ngày nhận lương: 5.
- Mua nhà | id: goal-1 | loại: house | mục tiêu: 100.000.000đ | hạn: 2031-09-15 | trạng thái: đang trả nợ | tài khoản mục tiêu: 2.818.000đ | giá trị hiện có: 2.818.000đ | giai đoạn trả nợ: đã tích luỹ + đã trả 3.969.000đ / tổng gốc và lãi 275.445.877đ | cần góp mỗi tháng: 1.151.000đ (tiền trả nợ) | trích tự động: 1.667.000đ/tháng vào ngày 6 (lần tới 2026-11-06) | còn khoản vay chưa trả hết
- Mua nhà | id: goal-8 | loại: house | mục tiêu: 100.000.000đ | hạn: 2027-09-15 | trạng thái: đang trả nợ | tài khoản mục tiêu: 6.858.000đ | giá trị hiện có: 6.858.000đ | giai đoạn trả nợ: đã tích luỹ + đã trả 8.334.000đ / tổng gốc và lãi 176.991.736đ | cần góp mỗi tháng: 1.476.000đ (ti

In [3]:
# 1. Tạo mục tiêu đủ thông tin -- skill goal-manage; có kế hoạch góp hằng tháng
ui = chat("Tôi muốn mua xe 600 triệu, cuối năm 2027, nạp trước 5 triệu.", "goal-1")["ui"]
show_ui(ui, "create_goal")
assert ui["target_date"] > goals["today"], "ngày phải sau hôm nay của hệ thống"
assert ui["initial_deposit"] <= goals["mainBalance"], "nạp ban đầu vượt số dư tài khoản chính"
if "monthly_amount" in ui:
    want = expected_monthly(ui)
    print(f"số góp agent đề xuất {vnd(ui['monthly_amount'])} | công thức {vnd(want)} | còn dư hằng tháng {goals['monthlySurplus']}")
    if abs(ui["monthly_amount"] - want) > 1000:
        print("(nhận xét nhẹ: số góp lệch công thức, khách vẫn sửa được trên form)")
else:
    print("(không có kế hoạch góp: chấp nhận được nếu câu trả lời nói rõ vì sao)")

Model call: 9.25s
Calling tool: read_file
Calling tool: get_goals
Finished tool: read_file (0.00s)
Finished tool: get_goals (1.02s)
Model call: 12.37s
Q: Tôi muốn mua xe 600 triệu, cuối năm 2027, nạp trước 5 triệu.
ANSWER:
Mình đề xuất mục tiêu **Mua xe 600.000.000đ**, hạn **31/12/2027**, nạp ban đầu **5.000.000đ**.

Để kịp hạn bạn cần góp khoảng **37.188.000đ mỗi tháng**, cao hơn khoản còn dư hằng tháng của bạn (12.840.000đ), nên bạn có thể cân nhắc lùi hạn hoặc giảm số tiền mục tiêu. Mình điền sẵn trích tự động vào ngày 6, bạn sửa được trước khi xác nhận.

Xác nhận xong mới mở tài khoản mục tiêu và trừ 5.000.000đ từ tài khoản chính.

UI:
{'type': 'create_goal', 'goal_name': 'Mua xe', 'template': 'car', 'target_amount': 600000000, 'target_date': '2027-12-31', 'initial_deposit': 5000000, 'monthly_amount': 37188000, 'monthly_day': 6, 'reason': 'Khách muốn mua xe 600.000.000đ vào cuối năm 2027 và nạp trước 5.000.000đ'}
------------------------------------------------------------
pfm sẽ g

In [4]:
# 2. Thời hạn tương đối: "trong 6 tháng" phải tính từ today của hệ thống, không phải ngày thật
ui = chat("Đặt mục tiêu du lịch 30 triệu trong 6 tháng.", "goal-2")["ui"]
show_ui(ui, "create_goal")
assert ui["template"] == "travel", ui["template"]
print("ngày agent chọn:", ui["target_date"], "| hôm nay của hệ thống:", goals["today"])

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError(\'HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Max retries exceeded with url: /info (Caused by ReadTimeoutError("HTTPSConnectionPool(host=\\\'api.smith.langchain.com\\\', port=443): Read timed out. (read timeout=10.0)"))\'))\nContent-Length: None\nAPI Key: lsv2_********************************************e2')
Run compression is not enabled. Please update to the latest version of LangSmith. Falling back to regular multipart ingestion.


Model call: 6.27s
Calling tool: read_file
Calling tool: get_goals
Finished tool: read_file (0.01s)
Finished tool: get_goals (0.90s)
Model call: 13.58s
Q: Đặt mục tiêu du lịch 30 triệu trong 6 tháng.
ANSWER:
Mình đề xuất mục tiêu **Du lịch 30.000.000đ**, hạn **21/04/2027**, chưa nạp trước.

Mỗi tháng bạn cần góp **5.000.000đ**, thấp hơn khoản còn dư hằng tháng của bạn (12.840.000đ) nên khá thoải mái. Mình điền sẵn trích tự động vào **ngày 6** (sau ngày nhận lương); bạn xem lại rồi xác nhận nhé.

Xác nhận xong mới mở tài khoản mục tiêu riêng và bắt đầu trích tiền hằng tháng.

UI:
{'type': 'create_goal', 'goal_name': 'Du lịch', 'template': 'travel', 'target_amount': 30000000, 'target_date': '2027-04-21', 'initial_deposit': 0, 'monthly_amount': 5000000, 'monthly_day': 6, 'reason': 'Khách muốn có 30.000.000đ cho chuyến du lịch trong 6 tháng tới'}
------------------------------------------------------------
pfm sẽ ghi khi khách xác nhận:
   {'type': 'createGoal', 'input': {'name': 'Du lịch',

In [ ]:
# 3. Thiếu số tiền và thời hạn: hỏi lại bằng NÚT (skill clarify-options), không trả create_goal và không hỏi bằng chữ tự do
body = chat("Tôi muốn đặt mục tiêu mua nhà.", "goal-3")
ui = body["ui"]
assert ui is not None and ui["type"] == "clarify_options", f"cần clarify_options, nhận {ui and ui['type']}"
assert is_valid_ui(ui), "form hỏi lại sai format"
questions = ui["questions"]
print("số câu hỏi:", len(questions), "| lựa chọn:", [q["options"] for q in questions])
assert len(questions) == 2, "thiếu cả số tiền và thời hạn nên cần đúng 2 câu"
# Bấm hai nút như khách (câu trả lời được gửi lại như khách tự gõ) -> lượt sau phải ra create_goal
follow = chat(f"{questions[0]['options'][1]}. {questions[1]['options'][2]}.", "goal-3")
assert follow["ui"] is not None and follow["ui"]["type"] == "create_goal", follow["ui"]
print("create_goal:", follow["ui"]["target_amount"], follow["ui"]["target_date"], "| hôm nay của hệ thống:", goals["today"])


Model call: 15.37s
Q: Tôi muốn đặt mục tiêu mua nhà.
ANSWER:
Bạn đang có 2 mục tiêu mua nhà rồi (mục tiêu 100 triệu, hạn 2031 và mục tiêu 100 triệu, hạn 2027), nên mục tiêu mới mình sẽ đặt tên khác để bạn dễ phân biệt nhé.

Bạn muốn tiết kiệm **bao nhiêu tiền** cho căn nhà, và **dự kiến đạt vào khi nào**?

UI:
None
------------------------------------------------------------


AssertionError: cần clarify_options, nhận None

Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by ReadTimeoutError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Read timed out. (read timeout=3)"))'))
Content-Length: 75198
API Key: lsv2_********************************************e2trace=01a15349-0543-7a51-906d-917278763479,id=01a15349-0543-7a51-906d-917278763479; trace=01a15349-0543-7a51-906d-917278763479,id=01a15349-0546-7701-888a-aef15ed4e673; trace=01a15349-0543-7a51-906d-917278763479,id=01a15349-0547-78a0-aa48-4ebb706b1f00
Failed to multipart ingest runs: Connection error caused failure to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HT

In [ ]:
# 4. Thêm tài sản khai tay (xe máy) vào một mục tiêu có sẵn -- skill goal-asset
assert TARGET_GOAL, "cần ít nhất một mục tiêu: tạo ở ô 1 rồi chạy ô 8 để ghi, sau đó chạy lại ô Setup"
ui = chat(f"Thêm xe máy Honda giá 30 triệu vào mục tiêu {TARGET_GOAL['name']}, tính đủ.", "goal-4")["ui"]
show_ui(ui, "add_asset")
assert ui["category"] == "motorbike" and ui["value"] == 30_000_000 and ui["goal_id"] == TARGET_GOAL["id"], ui

In [ ]:
# 5. Thêm tài sản theo mã (cổ phiếu): agent tra get_goal_market, form có instrument_id + quantity và KHÔNG có value
assert TARGET_GOAL, "cần ít nhất một mục tiêu"
ui = chat(f"Thêm 1000 cổ phiếu ACB vào mục tiêu {TARGET_GOAL['name']}, tính 80%.", "goal-5")["ui"]
show_ui(ui, "add_asset")
assert ui["category"] == "stock" and "value" not in ui and ui["quantity"] == 1000 and ui["count_percent"] == 80, ui
print("mã:", ui["instrument_id"])

In [ ]:
# 6. Tiền gửi ở ngân hàng khác: form có accrual (gốc, lãi suất, ngày bắt đầu) và không có value
assert TARGET_GOAL, "cần ít nhất một mục tiêu"
ui = chat(f"Tôi gửi 50 triệu ở Vietcombank lãi 5,2% một năm từ ngày 1/6/2026, tính vào mục tiêu {TARGET_GOAL['name']}.", "goal-6")["ui"]
show_ui(ui, "add_asset")
assert ui["category"] == "deposit" and "value" not in ui and ui["accrual"]["principal"] == 50_000_000, ui

In [ ]:
# 7. Hỏi về kế hoạch góp: câu trả lời phải dùng số thật từ get_goals (cần góp mỗi tháng, trích tự động đang cài, còn dư hằng tháng)
assert TARGET_GOAL, "cần ít nhất một mục tiêu"
body = chat(f"Mục tiêu {TARGET_GOAL['name']} của tôi mỗi tháng cần góp bao nhiêu để kịp hạn?", "goal-7")
print("get_goals nói: cần góp", vnd(TARGET_GOAL["requiredMonthly"]), "| trích tự động:", TARGET_GOAL["standingOrder"])
if body["ui"] is not None:
    show_ui(body["ui"], "edit_goal")

In [ ]:
# 7b. Chạy cả 9 loại tài sản (mỗi loại có form riêng): nhà, xe, xe máy, khác | vàng, cổ phiếu, quỹ | tiền gửi, trái phiếu
#     Mỗi loại ~15-25 giây (giới hạn 5 request/phút của model). Đổi ONLY để chạy riêng một vài loại, ví dụ ONLY = {"bond"}.
ONLY = None
assert TARGET_GOAL, "cần ít nhất một mục tiêu"
GN = TARGET_GOAL["name"]
MATRIX = [
    ("house", f"Thêm căn hộ Q7 giá 2 tỷ vào mục tiêu {GN}, tính 50%.", dict(value=2_000_000_000, count_percent=50)),
    ("car", f"Thêm xe ô tô Mazda 3 giá 600 triệu vào mục tiêu {GN}, tính đủ.", dict(value=600_000_000, count_percent=100)),
    ("motorbike", f"Thêm xe máy Honda Vision giá 30 triệu vào mục tiêu {GN}, tính đủ.", dict(value=30_000_000, count_percent=100)),
    ("other", f"Thêm bộ sưu tập đồng hồ giá 80 triệu vào mục tiêu {GN}, tính 60%.", dict(value=80_000_000, count_percent=60)),
    ("gold", f"Tôi có 2 chỉ vàng nhẫn 9999, tính vào mục tiêu {GN}.", dict(quantity=2)),
    ("stock", f"Thêm 1000 cổ phiếu ACB vào mục tiêu {GN}, tính 80%.", dict(quantity=1000, count_percent=80)),
    ("external_fund", f"Tôi có 500 chứng chỉ quỹ đầu tư cổ phiếu Amber ở công ty khác, thêm vào mục tiêu {GN}, tính 70%.", dict(quantity=500, count_percent=70)),
    ("deposit", f"Tôi gửi 50 triệu ở Vietcombank lãi 5,2% một năm từ ngày 1/6/2026, đáo hạn 1/6/2027, tính vào mục tiêu {GN}.", dict(principal=50_000_000, rate_pa=5.2)),
    ("bond", f"Tôi mua trái phiếu doanh nghiệp ABC 100 triệu, lãi 9% một năm, từ 1/3/2026 đến 1/3/2029, tính 90% vào mục tiêu {GN}.", dict(principal=100_000_000, rate_pa=9, count_percent=90)),
]
report = []
for category, message, want in MATRIX:
    if ONLY and category not in ONLY:
        continue
    ui = chat(message, f"asset-{category}")["ui"]
    problems = []
    if ui is None:
        problems.append("không có form")
    else:
        if ui["type"] != "add_asset" or ui["category"] != category:
            problems.append(f"type/category = {ui['type']}/{ui.get('category')}")
        if ui.get("goal_id") != TARGET_GOAL["id"]:
            problems.append(f"goal_id = {ui.get('goal_id')}")
        if "count_percent" in want and ui.get("count_percent") != want["count_percent"]:
            problems.append(f"count_percent = {ui.get('count_percent')}")
        if "value" in want and ui.get("value") != want["value"]:
            problems.append(f"value = {ui.get('value')}")
        if category in {"gold", "stock", "external_fund"}:
            if "value" in ui:
                problems.append("có value (hệ thống tự tính = số lượng x giá)")
            if not ui.get("instrument_id"):
                problems.append("thiếu instrument_id")
            if ui.get("quantity") != want["quantity"]:
                problems.append(f"quantity = {ui.get('quantity')}")
        if category in {"deposit", "bond"}:
            accrual = ui.get("accrual") or {}
            if "value" in ui:
                problems.append("có value (hệ thống tự tính = gốc + lãi)")
            if accrual.get("principal") != want["principal"] or accrual.get("rate_pa") != want["rate_pa"] or not accrual.get("start_date"):
                problems.append(f"accrual = {accrual}")
    report.append((category, problems))
    print(f">>> {category}: {'OK' if not problems else 'LỖI ' + '; '.join(problems)}\n")
print("Tổng kết:", {c: ("OK" if not p else "LỖI") for c, p in report})

## Option Q - test hard asset questions (add, edit, remove)

Same setup as Option P (run its setup cell first). Three cells:

- **9 - hard questions when ADDING an asset:** missing or ambiguous information ("20 triệu vàng", "quỹ An Bình", a deposit without rate and date, 1000,5 shares,
  a code that is not in the market list). Expect the agent to **ask back** and never guess: buttons (`clarify_options`) when the customer must pick one of 2-4 real choices
  (which gold, which fund, which goal), plain text when a number or a date is missing (quantity, price, rate, start date).
- **10a / 10b - EDIT and REMOVE an asset:** 10a puts five sample assets in memory only (nothing is written to `pfm`), 10b asks to change a hand-valued price, a gold / stock
  quantity, to remove an asset, to give a "new price" to a self-valued asset (expect NO `revalue`), to remove an MSB product (read-only: no form), and a bare "giờ tôi có 1500 cổ phiếu ACB"
  (the agent must read `get_goals` and propose editing the asset it already finds).

In [ ]:
# --- Bộ chạy câu hỏi (dùng cho các ô 9, 10, 11 bên dưới) ---
import copy
import time


def _no_form(ui):
    return ui is None or ui.get("type") not in {"add_asset", "edit_asset", "create_goal", "edit_goal", "close_goal"}


def NO_FORM(ui):
    return None if _no_form(ui) else f"không được trả form ({ui['type']})"


def ADD(category, **want):
    def check(ui):
        if ui is None or ui.get("type") != "add_asset":
            return "không có add_asset"
        if ui.get("category") != category:
            return f"category = {ui.get('category')}"
        for key, value in want.items():
            if value is ...:
                if key in ui:
                    return f"không được có {key}"
            elif ui.get(key) != value:
                return f"{key} = {ui.get(key)!r} (cần {value!r})"
    return check


def EDIT_ASSET(asset_id, change, **want):
    def check(ui):
        if ui is None or ui.get("type") != "edit_asset":
            return "không có edit_asset"
        if ui.get("asset_id") != asset_id or ui.get("change") != change:
            return f"asset_id/change = {ui.get('asset_id')}/{ui.get('change')}"
        for key, value in want.items():
            if ui.get(key) != value:
                return f"{key} = {ui.get(key)!r} (cần {value!r})"
    return check


def NOT_REVALUE(ui):
    if ui is not None and ui.get("type") == "edit_asset" and ui.get("change") == "revalue":
        return "đề xuất revalue cho tài sản tự định giá (giá do thị trường / lãi tính)"


def EDIT_GOAL(goal_id, **want):
    def check(ui):
        if ui is None or ui.get("type") != "edit_goal":
            return "không có edit_goal"
        if ui.get("goal_id") != goal_id:
            return f"goal_id = {ui.get('goal_id')}"
        for key, value in want.items():
            if value is ...:
                if key not in ui:
                    return f"thiếu {key}"
            elif value is None:
                if key in ui:
                    return f"không được có {key}"
            elif ui.get(key) != value:
                return f"{key} = {ui.get(key)!r} (cần {value!r})"
    return check


def run_cases(cases, only=None, prefix="case"):
    """cases: [(tên, câu hỏi, hàm kiểm tra)]. Lỗi HTTP không được tính là đạt. Model không chắc chắn: chạy lại nếu một ca lỗi."""
    report = []
    for i, (name, message, check) in enumerate(cases):
        if only and name.split()[0] not in only:
            continue
        started = time.time()
        response = client.post("/chat", headers=HEADERS, json={"message": message, "user_id": CUSTOMER, "thread_id": f"{prefix}-{i}"})
        if response.status_code != 200:
            problem, ui = f"lỗi HTTP {response.status_code} (không phải câu trả lời của agent)", None
            print(f"Q: {message}\n{response.text}\n" + "-" * 60)
        else:
            body = response.json()
            ui = body["ui"]
            print(f"Q: {message}\nANSWER:\n{body['answer']}\n\nUI:\n{ui}\n" + "-" * 60)
            problem = check(ui)
            if ui is not None and not is_valid_ui(ui):
                problem = (problem or "") + " | form sai format"
        report.append((name, problem))
        print(f">>> [{name}] {'OK' if not problem else 'LỖI: ' + problem} ({round(time.time() - started)}s)\n")
    print("Tổng kết:", sum(1 for _, p in report if not p), "/", len(report), "đạt")
    for name, problem in report:
        if problem:
            print("  LỖI", name, "->", problem)
    return report

In [ ]:
# 9. Câu hỏi khó khi THÊM tài sản: thiếu thông tin hoặc mơ hồ. Mong đợi: hỏi lại (nút hoặc chữ) hoặc form đúng quy tắc, không bao giờ đoán.
#    ONLY = {"A", "C"} để chạy riêng vài ca. Dòng "Câu hỏi lại" trong skill: nút khi chọn trong 2-4 phương án thật, chữ khi cần một con số / ngày.
assert TARGET_GOAL, "cần ít nhất một mục tiêu"
GN = TARGET_GOAL["name"]
ONLY = None
HARD_ADD = [
    ("A mua 20 triệu vàng: phải hỏi loại vàng (nút) và số lượng", f"Thêm 20 triệu vàng vào mục tiêu {GN}.", NO_FORM),
    ("B quỹ An Bình mơ hồ: hỏi quỹ nào (nút)", f"Thêm chứng chỉ quỹ An Bình vào mục tiêu {GN}, 100 đơn vị.",
     lambda ui: None if _no_form(ui) or ui.get("instrument_id") in {"fund-ABBF", "fund-ABEF"} else f"chọn mã lạ {ui.get('instrument_id')}"),
    ("C gửi tiền thiếu lãi suất và ngày: hỏi bằng chữ", f"Tôi gửi 50 triệu ở Techcombank, tính vào mục tiêu {GN}.", NO_FORM),
    ("D cổ phiếu số lẻ: không được đề xuất 1000,5", f"Thêm 1000,5 cổ phiếu ACB vào mục tiêu {GN}.",
     lambda ui: None if _no_form(ui) or (ui.get("type") == "add_asset" and ui.get("quantity") == int(ui["quantity"])) else "số lẻ cho cổ phiếu"),
    ("E vàng kèm giá khách nói: không được có value", f"Thêm 3 chỉ vàng SJC vào mục tiêu {GN}, mua giá 8 triệu một chỉ.", ADD("gold", value=...)),
    ("F nhà không nói giá: hỏi giá, không tự định giá", f"Thêm một căn nhà đang có vào mục tiêu {GN}.", NO_FORM),
    ("G xe máy không nói tỷ lệ: mặc định 100", f"Thêm xe máy Yamaha Exciter giá 40 triệu vào mục tiêu {GN}.", ADD("motorbike", value=40_000_000, count_percent=100)),
    ("H mã cổ phiếu không có trong danh mục: không bịa mã", f"Thêm 500 cổ phiếu XYZQ vào mục tiêu {GN}.",
     lambda ui: None if _no_form(ui) or ui.get("category") != "stock" else f"đặt mã tự bịa {ui.get('instrument_id')}"),
]
run_cases(HARD_ADD, ONLY, "hard-add")

In [ ]:
# 10a. Tài sản MẪU để thử sửa / bỏ (chỉ nằm trong bộ nhớ của notebook, KHÔNG ghi vào pfm).
#      Thêm vào mục tiêu TARGET_GOAL: xe máy khai tay, vàng, cổ phiếu ACB, tiền gửi Vietcombank (tự định giá) và một khoản tiền gửi MSB (chỉ xem).
INJECT_SAMPLE_ASSETS = True
if INJECT_SAMPLE_ASSETS:
    assert TARGET_GOAL, "cần ít nhất một mục tiêu"
    real_fetch = goal_tools.fetch_goals_snapshot if not hasattr(goal_tools, "_real_fetch") else goal_tools._real_fetch
    goal_tools._real_fetch = real_fetch

    def with_sample_assets(customer_id):
        snapshot = copy.deepcopy(real_fetch(customer_id))
        base = {"goalId": TARGET_GOAL["id"], "costBasis": 0, "valuationDate": snapshot["world"]["clock"]["today"], "countPercent": 100,
                "source": "self_reported", "status": "held", "createdAt": snapshot["world"]["clock"]["today"]}
        snapshot["world"]["assets"] += [
            {**base, "id": "asset-901", "type": "declared", "category": "motorbike", "name": "Xe máy Honda", "value": 30_000_000},
            {**base, "id": "asset-902", "type": "declared", "category": "gold", "name": "Vàng nhẫn", "value": 0, "instrumentId": "gold-ring", "quantity": 2, "source": "estimated"},
            {**base, "id": "asset-903", "type": "declared", "category": "stock", "name": "Cổ phiếu ACB", "value": 0, "instrumentId": "stock-acb", "quantity": 1000, "source": "estimated"},
            {**base, "id": "asset-904", "type": "declared", "category": "deposit", "name": "Gửi Vietcombank", "value": 50_000_000, "source": "estimated", "institution": "Vietcombank",
             "accrual": {"principal": 50_000_000, "ratePa": 5.2, "startDate": "2026-06-01", "maturityDate": "2027-06-01"}},
            {**base, "id": "asset-905", "type": "msb", "category": "term_deposit", "productId": "msb-term-deposit", "name": "Tiền gửi MSB 12 tháng", "value": 20_000_000, "source": "msb",
             "principal": 20_000_000, "ratePa": 5, "maturityDate": "2027-06-01"},
        ]
        return snapshot

    goal_tools.fetch_goals_snapshot = with_sample_assets
    goals = summarize_goals(goal_tools.fetch_goals_snapshot(CUSTOMER))
    print(render_goals(goals))
else:
    goal_tools.fetch_goals_snapshot = getattr(goal_tools, "_real_fetch", goal_tools.fetch_goals_snapshot)
    print("Đã bỏ tài sản mẫu.")

In [ ]:
# 10b. SỬA và BỎ tài sản (cần chạy ô 10a trước). Mong đợi: tài sản khai tay sửa được giá; vàng, cổ phiếu chỉ đổi số lượng;
#      tài sản tự định giá (vàng, tiền gửi) KHÔNG nhận "giá mới"; tài sản MSB chỉ xem, hướng dẫn vào màn Mục tiêu.
ONLY = None
EDIT_CASES = [
    ("I sửa giá xe máy khai tay", "Xe máy Honda của tôi giờ chỉ còn 28 triệu, cập nhật giúp.", EDIT_ASSET("asset-901", "revalue", value=28_000_000)),
    ("J đổi số lượng vàng", "Tôi mua thêm, giờ có 3 chỉ vàng nhẫn trong mục tiêu.", EDIT_ASSET("asset-902", "quantity", quantity=3)),
    ("K chỉ nói số cổ phiếu mới, không nhắc mục tiêu: phải tra rồi đề xuất sửa", "Giờ tôi có 1500 cổ phiếu ACB.", EDIT_ASSET("asset-903", "quantity", quantity=1500)),
    ("L đưa giá mới cho vàng: không được revalue", "Vàng của tôi giờ giá 9 triệu một chỉ, cập nhật giúp.", NOT_REVALUE),
    ("M bỏ cổ phiếu ACB", f"Bỏ cổ phiếu ACB khỏi mục tiêu {TARGET_GOAL['name']}.", EDIT_ASSET("asset-903", "remove")),
    ("N bỏ khoản gửi Vietcombank", f"Bỏ khoản gửi Vietcombank khỏi mục tiêu {TARGET_GOAL['name']}.", EDIT_ASSET("asset-904", "remove")),
    ("O bỏ tài sản MSB: không có form, chỉ hướng dẫn", f"Bỏ khoản tiền gửi MSB 12 tháng khỏi mục tiêu {TARGET_GOAL['name']}.", NO_FORM),
    ("P đưa giá mới cho khoản gửi: không được revalue", "Khoản gửi Vietcombank của tôi giờ là 60 triệu, cập nhật giúp.", NOT_REVALUE),
    ("Q thêm 3 chỉ vàng khi đã có 2 chỉ: cộng dồn thành 5", "Thêm 3 chỉ vàng nhẫn nữa vào mục tiêu.", EDIT_ASSET("asset-902", "quantity", quantity=5)),
]
run_cases(EDIT_CASES, ONLY, "edit-asset")

## Option R - test editing and closing a goal, and the monthly plan (`edit_goal`, `close_goal`)

Same setup as Option P, and it uses the runner cell of Option Q (`run_cases`): run those first. Questions on a goal the customer already has: move the date, change the amount (a goal that already has an auto-debit must come back with the **recalculated**
monthly amount, `monthly_amount` + `monthly_day`), change only the monthly plan, ask how much to save each month, a vague "đổi mục tiêu" (must ask back), close a goal (a goal with an
open loan cannot be closed: no form), and a goal that does not exist (no invented id). The last cell is **optional and writes to `pfm`**: it confirms the last proposal like the customer's button
(off by default, `APPLY = False`; point `PFM_URL` at your local pfm before turning it on).

In [ ]:
# 11. Mục tiêu có sẵn: lùi hạn, đổi số tiền, đổi kế hoạch góp, đóng, và câu hỏi mơ hồ. Mong đợi:
#     - đổi hạn / số tiền cho mục tiêu đang trích tự động thì form kèm số góp TÍNH LẠI (monthly_amount + monthly_day)
#     - chỉ đổi kế hoạch góp thì form chỉ có monthly_amount + monthly_day
#     - mục tiêu còn khoản vay thì không đóng được (không có form)
assert TARGET_GOAL, "cần ít nhất một mục tiêu"
GN, GID = TARGET_GOAL["name"], TARGET_GOAL["id"]
has_auto = bool(TARGET_GOAL["standingOrder"] and TARGET_GOAL["standingOrder"]["active"])
ONLY = None
GOAL_CASES = [
    ("R lùi hạn thêm 6 tháng", f"Lùi hạn mục tiêu {GN} thêm 6 tháng.", EDIT_GOAL(GID, new_target_date=..., **({"monthly_amount": ...} if has_auto else {}))),
    ("S giảm số tiền xuống 80 triệu", f"Giảm mục tiêu {GN} xuống còn 80 triệu.", EDIT_GOAL(GID, new_target_amount=80_000_000, **({"monthly_amount": ...} if has_auto else {}))),
    ("T chỉ đổi kế hoạch góp", f"Tăng số góp mục tiêu {GN} lên 6 triệu mỗi tháng, trích vào ngày 10.", EDIT_GOAL(GID, monthly_amount=6_000_000, monthly_day=10, new_target_date=None, new_target_amount=None)),
    ("U hỏi cần góp bao nhiêu: trả lời bằng số thật, không bắt buộc có form", f"Mục tiêu {GN} của tôi mỗi tháng cần góp bao nhiêu để kịp hạn?", lambda ui: None),
    ("V đổi mục tiêu mà không nói số hay ngày: hỏi lại", f"Tôi muốn đổi mục tiêu {GN}.", NO_FORM),
    ("W đóng mục tiêu (mục tiêu còn vay thì không có form)", f"Đóng mục tiêu {GN} giúp tôi.",
     lambda ui: None if (TARGET_GOAL["hasActiveLoan"] and _no_form(ui)) or (not TARGET_GOAL["hasActiveLoan"] and ui is not None and ui["type"] == "close_goal") else "kết quả không khớp trạng thái vay của mục tiêu"),
    ("X mục tiêu không có trong danh sách: không bịa id", "Lùi hạn mục tiêu mua tàu vũ trụ sang năm sau.", NO_FORM),
]
run_cases(GOAL_CASES, ONLY, "goal-edit")

In [ ]:
# 8. (TUỲ CHỌN, GHI DỮ LIỆU) Xác nhận đề xuất gần nhất giống khách bấm nút trên thẻ: gửi đúng các action tới pfm
#    Mặc định tắt. Nó GHI vào server pfm ở trên (mục tiêu / tài sản thật của CIF_0001): đặt PFM_URL = "http://localhost:3000" ở ô Setup để thử trên máy.
APPLY = False
LAST_UI = ui  # đề xuất của ô gần nhất đã chạy; đổi thành ui của ô khác nếu muốn

if not APPLY:
    print("APPLY = False: chưa ghi gì. Đổi thành True để gửi tới", PFM)
else:
    import uuid

    for action in pfm_actions(LAST_UI):
        r = httpx.post(f"{PFM}/api/goals-v2/actions", json={"cif": CUSTOMER, "requestId": str(uuid.uuid4()), "action": action}, timeout=60)
        print(action["type"], "->", r.status_code, (r.json().get("error") if r.status_code == 422 else "ok"))
        assert r.status_code == 200, r.text

## Verification checklist (local, rows 1-9 and 11 only — row 10 needs Studio, see docs/superpowers/plans/2026-08-30-personal-card-agent-verification.md)

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

verify_agent = create_deep_agent(
    model=default_model(),
    tools=[],
    middleware=[log_tool_calls, SeedCustomerContext()],
    system_prompt=instructions,
    checkpointer=MemorySaver(),
)

CHECKLIST_QUESTIONS = [
    "Tháng này tôi đã chi tiêu bao nhiêu trên thẻ?",
    "Giao dịch lớn nhất tuần vừa rồi là gì?",
    "Liệt kê các giao dịch trên 2 triệu trong tháng 7",
    'Danh mục nào đang "ngốn" tiền của tôi nhất?',
    "Tôi có đang trả lãi thẻ tín dụng không, và vì sao?",
    "Tôi nên dùng thẻ tín dụng hay ghi nợ cho khoản chi 5 triệu này?",
    "Ngày sao kê và ngày đến hạn thanh toán của thẻ tôi là khi nào?",
    "Tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Ngân sách của tôi là 15 triệu/tháng.",
    "Vậy tôi có đang chi tiêu vượt ngân sách tháng này không?",
    "Khoá thẻ của tôi giúp tôi.",
]
THREAD_CONFIG = {"configurable": {"thread_id": "notebook-verify-1"}}

for question in CHECKLIST_QUESTIONS:
    result = verify_agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config=THREAD_CONFIG,
    )
    print(f"Q: {question}\nA: {extract_final_text(result['messages'][-1].content)}\n{'-' * 60}")

print(
    "Row 10 (cross-thread durable budget memory) cannot be tested locally — "
    "run mda dev . and re-ask the budget question in a NEW Studio thread instead."
)
